# Day 10 — N-Gram Analysis

Many real skills are multi-word phrases (`machine learning`, `power bi`, `natural language processing`). A single-word (unigram) tokenizer misses these. This notebook generates unigrams, bigrams and trigrams and shows which multi-word skills get discovered.

In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

df = pd.read_csv('../data/clean_jobs.csv')

vectorizer = TfidfVectorizer(
    ngram_range=(1, 3),
    max_features=2000,
    stop_words='english'
)
X = vectorizer.fit_transform(df['clean_description'])
print(X.shape)

(10000, 2000)


## Rank n-grams by TF-IDF score

In [3]:
import numpy as np

scores = np.asarray(X.sum(axis=0)).ravel()
terms = vectorizer.get_feature_names_out()
ranking = pd.DataFrame({'term': terms, 'score': scores})
ranking['n_words'] = ranking['term'].str.split().apply(len)
ranking = ranking.sort_values('score', ascending=False)
ranking.head(20)

,term,score,n_words
583,experience,675.824326,1
424,data,607.100716,1
268,business,371.483346,1
1491,sql,350.110482,1
408,communication,337.912163,1
1337,required,337.912163,1
1332,reports,337.912163,1
1349,requires years,337.912163,2
1356,role,337.912163,1
1348,requires,337.912163,1


## Split out unigrams, bigrams, trigrams

In [4]:
unigrams = ranking[ranking['n_words'] == 1].head(15)
bigrams = ranking[ranking['n_words'] == 2].head(15)
trigrams = ranking[ranking['n_words'] == 3].head(15)

print('--- Top Unigrams ---')
print(unigrams['term'].tolist())
print('\n--- Top Bigrams ---')
print(bigrams['term'].tolist())
print('\n--- Top Trigrams ---')
print(trigrams['term'].tolist())

--- Top Unigrams ---
['experience', 'data', 'business', 'sql', 'communication', 'required', 'reports', 'role', 'requires', 'years', 'requirements', 'bi', 'power', 'python', 'solutions']

--- Top Bigrams ---
['requires years', 'communication attention', 'teams analyzing', 'preparing reports', 'cross functional', 'hands experience', 'position based', 'processes candidates', 'strong analytical', 'requirements preparing', 'required position', 'business processes', 'reports improving', 'candidates hands', 'role involves']

--- Top Trigrams ---
['processes candidates hands', 'requirements preparing reports', 'strong analytical thinking', 'teams analyzing requirements', 'preparing reports improving', 'analytical thinking communication', 'thinking communication attention', 'communication attention required', 'cross functional teams', 'required position based', 'reports improving business', 'business processes candidates', 'role involves working', 'analyzing requirements preparing', 'candidates

## Multi-word skills recovered that unigrams alone would have missed

In [5]:
known_multiword = ['machine learning', 'power bi', 'data science', 'natural language processing',
                    'data analysis', 'product analytics', 'business analysis',
                    'a b testing', 'requirements gathering', 'financial modeling',
                    'database administration', 'automation testing', 'cloud computing',
                    'marketing analytics', 'hr analytics', 'data quality',
                    'scikit learn', 'rest api']

found = ranking[ranking['term'].isin(known_multiword)][['term','score','n_words']]
found.sort_values('score', ascending=False)

,term,score,n_words
436,data quality,276.651656,2
1254,power bi,274.719580,2
1017,machine learning,165.412094,2
445,database administration,68.161471,2
772,hr analytics,68.062519,2
425,data analysis,68.062519,2
190,automation testing,67.593343,2
1343,requirements gathering,67.038872,2
269,business analysis,67.038872,2
337,cloud computing,66.380629,2


## Summary
- Generated unigrams, bigrams and trigrams with `TfidfVectorizer(ngram_range=(1,3))`
- Confirmed that key multi-word skills (`machine learning`, `power bi`, `data science`, `natural language processing`, etc.) only surface once bigrams/trigrams are included — they are invisible to a unigram-only pipeline
- Combined with the Day 8 phrase matcher and Day 9 TF-IDF ranking, n-grams close the loop on multi-word skill discovery

**Deliverable:** `Ngram_Analysis.ipynb` (this notebook).